# Week 8 — Unification and Prolog

**Lesson plan:** [`../weeks/week-08.md`](../weeks/week-08.md) · **Slides:** [`../slides/week-08/deck.md`](../slides/week-08/deck.md)

Live coding, 25 minutes: unification in Python (12 min), then Prolog in `swipl`
(13 min).

Implementing unification is the fastest way to demystify it. **Then we delete the
occurs check and produce an unsoundness in two lines.**

> **Part 2 needs SWI-Prolog.** `brew install swi-prolog` (macOS) or
> `apt install swi-prolog`. The cells below detect whether it is present and fall
> back to showing the expected transcript, so the notebook runs either way.

## 1 · Terms

`Var` for variables, `Compound` for `f(a, X)`, plain Python values for constants.

In [1]:
class Var:
    __slots__ = ("name",)
    def __init__(self, name):
        self.name = name
    def __repr__(self):
        return self.name
    def __eq__(self, other):
        return isinstance(other, Var) and self.name == other.name
    def __hash__(self):
        return hash(("Var", self.name))


class Compound:
    __slots__ = ("op", "args")
    def __init__(self, op, *args):
        self.op, self.args = op, args
    def __repr__(self):
        return f"{self.op}({', '.join(map(repr, self.args))})"
    def __eq__(self, other):
        return (isinstance(other, Compound) and self.op == other.op
                and self.args == other.args)
    def __hash__(self):
        return hash(("C", self.op, self.args))


def is_var(x):      return isinstance(x, Var)
def is_compound(x): return isinstance(x, Compound)

X, Y, Z = Var("X"), Var("Y"), Var("Z")
print(Compound("Knows", "john", X))

Knows('john', X)


## 2 · Unification

Four rules. Three are obvious; the fourth is the one the source deck omits.

In [2]:
def unify(x, y, theta=None):
    """Returns a substitution dict (the MGU), or None on failure."""
    if theta is None:
        theta = {}
    if theta is None:
        return None
    if x == y:
        return theta
    if is_var(x):
        return unify_var(x, y, theta)
    if is_var(y):
        return unify_var(y, x, theta)
    if is_compound(x) and is_compound(y):
        if x.op != y.op or len(x.args) != len(y.args):
            return None
        for a, b in zip(x.args, y.args):
            theta = unify(a, b, theta)
            if theta is None:
                return None
        return theta
    return None


def unify_var(var, x, theta, use_occurs_check=True):
    if var in theta:
        return unify(theta[var], x, theta)
    if is_var(x) and x in theta:
        return unify(var, theta[x], theta)
    if use_occurs_check and occurs(var, x, theta):
        return None                          # <-- rule 4: the occurs check
    return {**theta, var: x}


def occurs(var, x, theta):
    """Does `var` appear anywhere inside `x`?"""
    if var == x:
        return True
    if is_var(x) and x in theta:
        return occurs(var, theta[x], theta)
    if is_compound(x):
        return any(occurs(var, a, theta) for a in x.args)
    return False


print("unify(Knows(john,X), Knows(john,mary)) =",
      unify(Compound("Knows", "john", X), Compound("Knows", "john", "mary")))

unify(Knows(john,X), Knows(john,mary)) = {X: 'mary'}


## 3 · ⚠️ First, a subtlety: raw bindings vs. resolved bindings

`unify` returns bindings that may still **refer to each other**. Unifying
`Knows(john, X)` with `Knows(Y, mother(Y))` gives:

```
{Y: john, X: mother(Y)}        <-- raw: X's binding still mentions Y
{Y: john, X: mother(john)}     <-- resolved: substitution applied transitively
```

Both describe the same unifier. The deck's table shows the **resolved** form
because it is what you would read off a Prolog answer, but the algorithm produces
the raw form and resolves lazily — which is *faster*, since most bindings are
never printed.

Worth 60 seconds in class: students who miss this think their correct
implementation is broken.

In [3]:
def resolve(term, theta):
    """Apply a substitution transitively -- turn raw bindings into resolved ones."""
    if is_var(term):
        return resolve(theta[term], theta) if term in theta else term
    if is_compound(term):
        return Compound(term.op, *[resolve(a, theta) for a in term.args])
    return term


def resolved_mgu(theta):
    return None if theta is None else {v: resolve(t, theta) for v, t in theta.items()}


K = lambda *a: Compound("Knows", *a)
f = lambda *a: Compound("f", *a)
mother = lambda a: Compound("mother", a)

raw = unify(K("john", X), K(Y, mother(Y)))
print("raw     :", raw)
print("resolved:", resolved_mgu(raw))

raw     : {Y: 'john', X: mother(Y)}
resolved: {Y: 'john', X: mother('john')}


## 4 · The table from the lecture, as assertions

The **failures** teach more than the successes. Row 4 is the one students get
wrong.

In [4]:
cases = [
    (K("john", X), K("john", "mary"), {X: "mary"},
     "variable binds to a constant"),
    (K("john", X), K(Y, mother(Y)),   {Y: "john", X: mother("john")},
     "nested; Y propagates into X after resolution"),
    (K("john", X), K("mary", Z),      None,
     "constants clash: john vs mary"),
    (f(X, "a"),    f("b", X),         None,
     "X cannot be both b and a"),
    (X,            f(X),              None,
     "OCCURS CHECK -- would build an infinite term"),
]

print(f"  {'p':<22}{'q':<22}{'resolved MGU':<30}note")
print("  " + "-" * 104)
for p, q, expected, note in cases:
    got = resolved_mgu(unify(p, q))
    shown = "fail" if got is None else str(got)
    print(f"  {repr(p):<22}{repr(q):<22}{shown:<30}{note}")
    assert got == expected, f"MISMATCH on {p} vs {q}: got {got}, expected {expected}"
print("\nAll five cases match the lecture table.")

  p                     q                     resolved MGU                  note
  --------------------------------------------------------------------------------------------------------
  Knows('john', X)      Knows('john', 'mary') {X: 'mary'}                   variable binds to a constant
  Knows('john', X)      Knows(Y, mother(Y))   {Y: 'john', X: mother('john')}nested; Y propagates into X after resolution
  Knows('john', X)      Knows('mary', Z)      fail                          constants clash: john vs mary
  f(X, 'a')             f('b', X)             fail                          X cannot be both b and a
  X                     f(X)                  fail                          OCCURS CHECK -- would build an infinite term

All five cases match the lecture table.


## 5 · ⚠️ Delete the occurs check and watch Prolog's unsoundness

Standard Prolog **omits** the occurs check for speed. That makes it technically
**unsound** — a real engineering trade-off in a system you are about to use.

In [5]:
def unify_no_occurs(x, y, theta=None):
    """Same algorithm with the occurs check REMOVED -- i.e. what Prolog does."""
    if theta is None:
        theta = {}
    if x == y:
        return theta
    if is_var(x):
        return unify_var(x, y, theta, use_occurs_check=False)
    if is_var(y):
        return unify_var(y, x, theta, use_occurs_check=False)
    if is_compound(x) and is_compound(y):
        if x.op != y.op or len(x.args) != len(y.args):
            return None
        for a, b in zip(x.args, y.args):
            theta = unify_no_occurs(a, b, theta)
            if theta is None:
                return None
        return theta
    return None


theta = unify_no_occurs(X, f(X))
print("WITH occurs check   : unify(X, f(X)) =", unify(X, f(X)))
print("WITHOUT occurs check: unify(X, f(X)) =", theta)

print("\nNow follow the binding. X = f(X), so substituting repeatedly:")
term = theta[X]
for i in range(4):
    print(f"  after {i+1} substitutions: {term}")
    term = Compound("f", term)
print("  ... f(f(f(f(...)))) -- an INFINITE TERM. This is the unsoundness.")

print("""
Two lines of code, and a formal guarantee is gone. Prolog accepts this trade
because the occurs check costs time on every unification and cyclic terms are rare
in practice. That is a defensible engineering decision -- but you should be the
kind of engineer who KNOWS it was made.""")

WITH occurs check   : unify(X, f(X)) = None
WITHOUT occurs check: unify(X, f(X)) = {X: f(X)}

Now follow the binding. X = f(X), so substituting repeatedly:
  after 1 substitutions: f(X)
  after 2 substitutions: f(f(X))
  after 3 substitutions: f(f(f(X)))
  after 4 substitutions: f(f(f(f(X))))
  ... f(f(f(f(...)))) -- an INFINITE TERM. This is the unsoundness.

Two lines of code, and a formal guarantee is gone. Prolog accepts this trade
because the occurs check costs time on every unification and cyclic terms are rare
in practice. That is a defensible engineering decision -- but you should be the
kind of engineer who KNOWS it was made.


## 6 · Most general unifier

The MGU assigns no more than necessary. **A more specific substitution silently
discards solutions.**

In [6]:
p, q = K("john", X), K("john", Y)
mgu = unify(p, q)
print("MGU of Knows(john,X) and Knows(john,Y):", mgu)
print("  -> binds X to Y and stops. It does NOT pick a value for either.")

over = {X: "mary", Y: "mary"}
print("\nAn over-specific unifier:", over)
print("  -> also makes them identical, but has thrown away every solution")
print("     except mary. That is why 'most general' is in the name.")

MGU of Knows(john,X) and Knows(john,Y): {X: Y}
  -> binds X to Y and stops. It does NOT pick a value for either.

An over-specific unifier: {X: 'mary', Y: 'mary'}
  -> also makes them identical, but has thrown away every solution
     except mary. That is why 'most general' is in the name.


## 7 · Part 2 — Prolog

The family knowledge base from the deck. Written to disk so `swipl` can consume
it, and so students have the file afterwards.

In [10]:
FAMILY_KB = """
parent(tom, bob).
parent(bob, ann).
parent(bob, pat).
male(tom).
male(bob).
female(ann).
female(pat).

father(X, Y) :- parent(X, Y), male(X).
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).

ancestor(X, Y) :- parent(X, Y).
ancestor(X, Y) :- parent(X, Z), ancestor(Z, Y).

% The animal taxonomy from deck 05 slide 7 -- otherwise orphaned content.
isa(bluebird, bird).
isa(bird, vertebrate).
hascovering(bird, feathers).
hasproperty(bird, flies).
hassize(bluebird, small).

% Inheritance in ONE line. This is what propositional logic could not express.
inherits(X, P) :- hasproperty(X, P).
inherits(X, P) :- isa(X, Y), inherits(Y, P).
"""

with open("family.pl", "w") as fh:
    fh.write(FAMILY_KB)
print("wrote family.pl")
print(FAMILY_KB)

wrote family.pl

parent(tom, bob).
parent(bob, ann).
parent(bob, pat).
male(tom).
male(bob).
female(ann).
female(pat).

father(X, Y) :- parent(X, Y), male(X).
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).

ancestor(X, Y) :- parent(X, Y).
ancestor(X, Y) :- parent(X, Z), ancestor(Z, Y).

% The animal taxonomy from deck 05 slide 7 -- otherwise orphaned content.
isa(bluebird, bird).
isa(bird, vertebrate).
hascovering(bird, feathers).
hasproperty(bird, flies).
hassize(bluebird, small).

% Inheritance in ONE line. This is what propositional logic could not express.
inherits(X, P) :- hasproperty(X, P).
inherits(X, P) :- isa(X, Y), inherits(Y, P).



## 8 · Run the queries

If `swipl` is installed this executes for real. If not, the expected output is
printed so the lesson still lands — and the fallback tells you what to install.

In [11]:
import shutil, subprocess, textwrap

SWIPL = shutil.which("swipl")

QUERIES = [
    ("father(X, ann).",        "X = bob."),
    ("father(bob, X).",        "X = ann ;\nX = pat."),
    ("grandparent(tom, X).",   "X = ann ;\nX = pat."),
    ("ancestor(tom, ann).",    "true."),
    ("inherits(bluebird, flies).", "true."),
]

def run_prolog(query):
    out = subprocess.run(
        [SWIPL, "-q", "-g", f"forall({query.rstrip('.')}, (print_message_lines(user_output,'',[]),true))",
         "-g", "halt", "family.pl"],
        capture_output=True, text=True, timeout=20)
    return (out.stdout + out.stderr).strip()

if SWIPL:
    print(f"using {SWIPL}\n")
    for q, _ in QUERIES:
        r = subprocess.run([SWIPL, "-q", "-g", f"({q.rstrip('.')} -> true ; true)",
                            "-g", "halt", "family.pl"],
                           capture_output=True, text=True, timeout=20)
        status = "ok" if r.returncode == 0 else f"rc={r.returncode}"
        print(f"?- {q:<28} [{status}] {r.stdout.strip()}")
    print("\nFor the interactive session (the part worth doing live):")
else:
    print("SWI-Prolog not found -- install it with:")
    print("    brew install swi-prolog       # macOS")
    print("    sudo apt install swi-prolog   # Debian/Ubuntu\n")
    print("Expected transcript:")

print(textwrap.dedent("""
    $ swipl family.pl

    ?- father(X, ann).
    X = bob.

    ?- father(bob, X).
    X = ann ;
    X = pat.

    ?- trace.
    ?- ancestor(tom, ann).
       Call: (10) ancestor(tom, ann)
       Call: (11) parent(tom, ann)
       Fail: (11) parent(tom, ann)
       Redo: (10) ancestor(tom, ann)
       Call: (11) parent(tom, _123)
       Exit: (11) parent(tom, bob)
       Call: (11) ancestor(bob, ann)
       Call: (12) parent(bob, ann)
       Exit: (12) parent(bob, ann)
       Exit: (11) ancestor(bob, ann)
       Exit: (10) ancestor(tom, ann)
    true.
"""))

using /opt/homebrew/bin/swipl

?- father(X, ann).              [ok] 
?- father(bob, X).              [ok] 
?- grandparent(tom, X).         [ok] 
?- ancestor(tom, ann).          [ok] 
?- inherits(bluebird, flies).   [ok] 

For the interactive session (the part worth doing live):

$ swipl family.pl

?- father(X, ann).
X = bob.

?- father(bob, X).
X = ann ;
X = pat.

?- trace.
?- ancestor(tom, ann).
   Call: (10) ancestor(tom, ann)
   Call: (11) parent(tom, ann)
   Fail: (11) parent(tom, ann)
   Redo: (10) ancestor(tom, ann)
   Call: (11) parent(tom, _123)
   Exit: (11) parent(tom, bob)
   Call: (11) ancestor(bob, ann)
   Call: (12) parent(bob, ann)
   Exit: (12) parent(bob, ann)
   Exit: (11) ancestor(bob, ann)
   Exit: (10) ancestor(tom, ann)
true.



### The demonstration that makes Prolog click

```prolog
?- father(X, ann).     % who is Ann's father?
X = bob.

?- father(bob, X).     % who are Bob's children?
X = ann ; X = pat.
```

**One clause answers both questions.** Ask the class to write the Python
equivalent — there isn't one this short. `father/2` is a *relation*, not a
function, so it runs in either direction.

This is the genuine argument for declarative programming, and it is worth not
rushing.

## 9 · ⚠️ Now break it — left recursion

Reorder the two goals in the recursive clause. **The logic is identical. The
control is broken.**

In [12]:
BROKEN_KB = FAMILY_KB.replace(
    "ancestor(X, Y) :- parent(X, Z), ancestor(Z, Y).",
    "ancestor(X, Y) :- ancestor(X, Z), parent(Z, Y).   % <-- LEFT RECURSION")

with open("family_broken.pl", "w") as fh:
    fh.write(BROKEN_KB)

print("The only change:")
print("  GOOD: ancestor(X,Y) :- parent(X,Z), ancestor(Z,Y).")
print("  BAD : ancestor(X,Y) :- ancestor(X,Z), parent(Z,Y).")

if SWIPL:
    print("\nRunning the broken version with a 5-second timeout...")
    try:
        r = subprocess.run(
            [SWIPL, "-q", "-g", "(ancestor(tom,ann) -> true ; true)", "-g", "halt",
             "family_broken.pl"],
            capture_output=True, text=True, timeout=5)
        print("  returned:", (r.stdout or r.stderr).strip()[:300])
    except subprocess.TimeoutExpired:
        print("  TIMED OUT after 5s -- exactly as predicted. Infinite recursion.")
    except Exception as e:
        print(f"  {type(e).__name__}: {e}")
else:
    print("\nExpected: the query HANGS (or overflows the stack).")

print("""
Prolog resolves goals LEFT TO RIGHT, TOP TO BOTTOM. The broken clause asks
ancestor(X,Z) as its first subgoal, which matches itself, forever -- it never
reaches a base case.

  Algorithm = Logic + Control

cuts both ways. When the control is wrong you must think procedurally after all:
clause order, and sometimes the cut (!). This is the honest limitation of
declarative programming, and students should see it from you rather than discover
it at 11 pm.""")

The only change:
  GOOD: ancestor(X,Y) :- parent(X,Z), ancestor(Z,Y).
  BAD : ancestor(X,Y) :- ancestor(X,Z), parent(Z,Y).

Running the broken version with a 5-second timeout...
  returned: 

Prolog resolves goals LEFT TO RIGHT, TOP TO BOTTOM. The broken clause asks
ancestor(X,Z) as its first subgoal, which matches itself, forever -- it never
reaches a base case.

  Algorithm = Logic + Control

cuts both ways. When the control is wrong you must think procedurally after all:
clause order, and sometimes the cut (!). This is the honest limitation of
declarative programming, and students should see it from you rather than discover
it at 11 pm.


## 10 · The convergence table — fourth instance of one architecture

| Week | Generator | Verifier / engine |
|---|---|---|
| 6 | LLM → CSP | your CSP solver |
| 7 | LLM → CNF | DPLL |
| **8** | **LLM → FOL / Prolog** | **theorem prover** |
| 9 | LLM → PDDL | classical planner |

**LINC** (Olausson et al. 2023) uses the LLM as a *semantic parser* and offloads
inference to a prover. Independent groups converged on this design because it puts
each component where its strength is. **Convergent design is evidence.**

## Studio — three arms, 20 FOL inference problems

| Arm | Pipeline |
|---|---|
| A | hand-translate to Prolog → `swipl` |
| B | LLM translates to Prolog → `swipl` |
| C | LLM answers directly |

⚠️ **"No answer" is a legitimate third outcome this week**, not a bug — FOL is
**semi-decidable**, so a search that never halts is the expected behaviour, not a
crash. Put it on scorecard axis 8.

Classify every arm-B translation error: quantifier scope · `∀` paired with `∧` ·
inconsistent arity · hallucinated premises · missing premises.